In [7]:
import sys
import os
import time
import json
import re
import math
from datetime import datetime

import pandas as pd
import numpy as np
from tqdm import tqdm
import krippendorff
# import matplotlib.pyplot as plt

# LLLM Annotation File Paths
LLM_ANNOTATION_FILES = {
    "openai": "csv/llm_annotations_openai.csv",
    # "anthropic": "csv/llm_annotations_anthropic.csv",
    # "deepseek": "csv/llm_annotations_deepseek.csv",
    "openai_mini": "csv/llm_annotations_openai_mini.csv"
}   

MODELS = {
    1: {
        "llm_name_version": "gpt-4o-2024-08-06",
        "provider":         "openai",
        "display_name":     "GPT-4o",
        "rationale":        "Benchmark; used in Jia et al. 2024, Törnberg 2023/2025, Gilardi 2023"
    },
    # 2: {
    #     "llm_name_version": "claude-haiku-4-5-20251001",
    #     "provider":         "anthropic",
    #     "display_name":     "claude-haiku-4-5",
    #     "rationale":        "not sure yet"
    # },
    # 3: {
    #     "llm_name_version": "deepseek-chat",
    #     "provider":         "deepseek",
    #     "display_name":     "DeepSeek-V3",
    #     "rationale":        "Cost-efficient frontier model with strong instruction-following and Chinese"
    # },
    4: {
        "llm_name_version": "gpt-4o-mini",
        "provider":         "openai",
        "display_name":     "GPT-4o-mini",
        "rationale":        "Comparison; OpenAI's smaller model, much weaker than GPT-4o"
    },
}

Load an inspect annotated dataset

In [8]:
for model_key, model_info in MODELS.items():
    provider = model_info["provider"]
    df = pd.read_csv(LLM_ANNOTATION_FILES[provider])

    required_cols = [
        "tweet_id", "llm_id", "llm_name_version",
        "V1", "V1_reason", "V2", "V2_reason", "V3", "V3_reason",
        "V4", "V4_reason", "V5", "V5_reason", "V6", "V6_reason",
        "V7", "V7_reason", "V8", "V8_reason",
        "total_score", "date", "annotation_session"
    ]
    missing = [c for c in required_cols if c not in df.columns]
    if missing:
        raise ValueError(f"Missing required columns: {missing}")

    print(f"Loaded {len(df)} tweets")
    print(f"\nColumn overview")
    print(df.dtypes)
    print(f"\nPost type distribution (1=original, 2=reply, 3=retweet):")
    print(df['post_type'].value_counts())
    print(f"\nSample tweets:")
    df[["tweet_id", "post_type", "text"]].head(3)

Loaded 301 tweets

Column overview
llm_annotation_id     int64
tweet_id              int64
llm_id                int64
llm_name_version        str
post_type               str
text                    str
V1                    int64
V1_reason               str
V2                    int64
V2_reason               str
V3                    int64
V3_reason               str
V4                    int64
V4_reason               str
V5                    int64
V5_reason               str
V6                    int64
V6_reason               str
V7                    int64
V7_reason               str
V8                    int64
V8_reason               str
total_score           int64
date                    str
annotation_session      str
dtype: object

Post type distribution (1=original, 2=reply, 3=retweet):
post_type
reply    201
tweet     60
quote     40
Name: count, dtype: int64

Sample tweets:
Loaded 301 tweets

Column overview
llm_annotation_id     int64
tweet_id              int64
llm_id     

In [9]:

SCORE_COLS = ["V1", "V2", "V3", "V4", "V5", "V6", "V7", "V8"]

# Load all annotation files and align by tweet_id
model_aligned = {}

for model_key, model_info in MODELS.items():
    provider = model_info["provider"]
    df = pd.read_csv(LLM_ANNOTATION_FILES[provider])
    df = df[["tweet_id"] + SCORE_COLS].dropna(subset=SCORE_COLS)
    df = df.set_index("tweet_id")
    model_aligned[model_info["display_name"]] = df

# Keep only tweet_ids present in ALL models
common_ids = set.intersection(*[set(df.index) for df in model_aligned.values()])
print(f"Tweets annotated by all models: {len(common_ids)}")

for name in model_aligned:
    model_aligned[name] = model_aligned[name].loc[list(common_ids)]

print("model_aligned ready:", list(model_aligned.keys()))

Tweets annotated by all models: 301
model_aligned ready: ['GPT-4o', 'GPT-4o-mini']


#### Inter-LLM Consistency (RQ2)
- Pairwise Krippendorff's alpha across all model pairs
- Calculating multi-rater alpha across all LLMs simultaneously

In [10]:
model_names = list(model_aligned.keys())
print("=" * 55)
print("Inter-LLM Consistency: Pairwise Krippendorff's alpha")
print("=" * 55)

inter_llm_records = []

for col in SCORE_COLS:
    print(f"\n  {col}")
    print(f"  {'Model A':<22} {'Model B':<22} {'alpha':>7}")
    print("  " + "-" * 55)

    for i in range(len(model_names)):
        for j in range(i + 1, len(model_names)):
            name_a, name_b = model_names[i], model_names[j]
            v_a = model_aligned[name_a][col].values
            v_b = model_aligned[name_b][col].values

            alpha = krippendorff.alpha(
                reliability_data=np.array([v_a, v_b], dtype=float),
                level_of_measurement='ordinal'
            )
            print(f"  {name_a:<22} {name_b:<22} {alpha:>7.3f}")
            inter_llm_records.append({
                "Variable": col,
                "Model_A":  name_a,
                "Model_B":  name_b,
                "Krippendorff_alpha": round(alpha, 4)
            })

    # Multi-rater alpha (all LLMs together)
    all_raters = np.array([
        model_aligned[name][col].values for name in model_names
    ], dtype=float)
    multi_alpha = krippendorff.alpha(
        reliability_data=all_raters,
        level_of_measurement='ordinal'
    )
    print(f"  {'Multi-rater (all LLMs)':<44} {multi_alpha:>7.3f}")

inter_llm_df = pd.DataFrame(inter_llm_records)
inter_llm_df.to_csv("csv/table_inter_llm_consistency_openai_comparison.csv", index=False)
print("\nSaved: table_inter_llm_consistency.csv")

Inter-LLM Consistency: Pairwise Krippendorff's alpha

  V1
  Model A                Model B                  alpha
  -------------------------------------------------------
  GPT-4o                 GPT-4o-mini              1.000
  Multi-rater (all LLMs)                         1.000

  V2
  Model A                Model B                  alpha
  -------------------------------------------------------
  GPT-4o                 GPT-4o-mini              1.000
  Multi-rater (all LLMs)                         1.000

  V3
  Model A                Model B                  alpha
  -------------------------------------------------------
  GPT-4o                 GPT-4o-mini              1.000
  Multi-rater (all LLMs)                         1.000

  V4
  Model A                Model B                  alpha
  -------------------------------------------------------
  GPT-4o                 GPT-4o-mini              1.000
  Multi-rater (all LLMs)                         1.000

  V5
  Model A        

H